# Module 9: Log budget alerts (job task 4)
Runs **only when** the If/else task found at least one category over budget.
Appends those categories to an alerts table, building a history of every overspend.

In [0]:
dbutils.widgets.text("schema", "finance_pipeline", "Schema with the gold tables")
schema = dbutils.widgets.get("schema")

alerts_table = "workspace.finance.budget_alerts"
source = f"workspace.{schema}.gold_budget_status"
latest = f"(SELECT max(month) FROM {source})"

In [0]:
# Create the alerts table the first time (same columns as gold_budget_status + a timestamp)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {alerts_table} AS
    SELECT current_timestamp() AS alert_time, * FROM {source} WHERE 1 = 0
""")

# Append this run's overspends
spark.sql(f"""
    INSERT INTO {alerts_table}
    SELECT current_timestamp(), * FROM {source}
    WHERE status = 'OVER' AND month = {latest}
""")

display(spark.sql(f"SELECT * FROM {alerts_table} ORDER BY alert_time DESC"))